# FactFlow Demo Notebook

This notebook demonstrates the FactFlow agent system for sentiment-reality analysis.

## What is FactFlow?

FactFlow is a multi-agent system that compares news sentiment with actual market price action to identify market inefficiencies. It protects users from "Fake News" dumps and "Hollow Hype" pumps by providing data-driven market analysis.

### Architecture

FactFlow uses a **Sequential Multi-Agent System** with three specialized agents:

1. **News Scout Agent** - Analyzes news sentiment using Google Search
2. **Market Analyst Agent** - Fetches real-time market data (price, volume, market cap)
3. **Judge Agent** - Synthesizes findings and provides recommendations

```
User Query → News Scout → Market Analyst → Judge → Final Recommendation
```

### Key Features

- **Divergence Detection**: Identifies when sentiment and price action diverge
- **Real-Time Data**: Integration with live market APIs
- **Session Management**: Multi-turn conversations with context
- **Memory Bank**: Long-term storage of historical analyses
- **Observability**: Logging, tracing, and metrics collection


In [8]:
# Install required dependencies if not already installed
import subprocess
import sys

# Map of package names to import names (for packages with different import names)
package_import_map = {
    "google-adk": "google.adk",
    "google-generativeai": "google.generativeai",
    "yfinance": "yfinance",
    "python-dotenv": "dotenv",
    "requests": "requests",
    "pandas": "pandas",
    "numpy": "numpy",
}

required_packages = [
    "google-adk>=0.1.0",
    "google-generativeai>=0.8.0",
    "yfinance>=0.2.0",
    "requests>=2.31.0",
    "python-dotenv>=1.0.0",
    "pandas>=2.0.0",
    "numpy>=1.24.0",
]

def check_and_install_package(package):
    """Check if a package is installed, install if not."""
    package_name = package.split(">=")[0].split("==")[0]
    import_name = package_import_map.get(package_name, package_name.replace("-", "_"))
    
    try:
        __import__(import_name)
        print(f"✅ {package_name} is already installed")
        return True
    except ImportError:
        print(f"📦 Installing {package}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package], 
                                stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
            # Verify installation
            try:
                __import__(import_name)
                print(f"✅ {package_name} installed successfully")
                return True
            except ImportError:
                print(f"⚠️  {package_name} installed but import failed")
                return False
        except subprocess.CalledProcessError as e:
            print(f"❌ Failed to install {package_name}")
            return False

print("Checking dependencies...")
print("=" * 60)
missing_packages = []
for package in required_packages:
    package_name = package.split(">=")[0].split("==")[0]
    if not check_and_install_package(package):
        missing_packages.append(package_name)

print("=" * 60)
if missing_packages:
    print(f"\n⚠️  Warning: Some packages could not be installed: {', '.join(missing_packages)}")
    print("   Please install them manually using:")
    print(f"   pip install {' '.join(missing_packages)}")
    print("\n   Or install all requirements:")
    print("   pip install -r factflow/requirements.txt")
else:
    print("\n✅ All dependencies are installed!")


Checking dependencies...
✅ google-adk is already installed
📦 Installing google-generativeai>=0.8.0...


C:\Users\Soroush-PC\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ google-generativeai installed successfully
📦 Installing yfinance>=0.2.0...
✅ yfinance installed successfully
✅ requests is already installed
✅ python-dotenv is already installed
✅ pandas is already installed
✅ numpy is already installed

✅ All dependencies are installed!


## Setup

First, let's set up the environment and import necessary modules.


In [9]:
import os
import sys
import asyncio
from pathlib import Path
from dotenv import load_dotenv

# Add the directory containing the factflow package to Python path
current_dir = Path.cwd()
print(f"Current working directory: {current_dir}")

# If we're in the notebooks directory, we need to go up to the parent that contains factflow
if current_dir.name == "notebooks":
    # We're in factflow/notebooks/, so factflow/ is the parent
    factflow_dir = current_dir.parent
    # The directory containing factflow/ needs to be in the path
    parent_of_factflow = factflow_dir.parent
    if str(parent_of_factflow) not in sys.path:
        sys.path.insert(0, str(parent_of_factflow))
    print(f"Added to path: {parent_of_factflow}")
elif current_dir.name == "factflow":
    # We're in factflow/, so add the parent directory
    parent_dir = current_dir.parent
    if str(parent_dir) not in sys.path:
        sys.path.insert(0, str(parent_dir))
    print(f"Added to path: {parent_dir}")
else:
    # Try to find factflow directory
    factflow_path = current_dir / "factflow"
    if factflow_path.exists():
        if str(current_dir) not in sys.path:
            sys.path.insert(0, str(current_dir))
        print(f"Added to path: {current_dir}")
    else:
        # Look for factflow in parent directories
        for parent in current_dir.parents:
            factflow_check = parent / "factflow"
            if factflow_check.exists() and factflow_check.is_dir():
                if str(parent) not in sys.path:
                    sys.path.insert(0, str(parent))
                print(f"Added to path: {parent}")
                break

# Load environment variables - try multiple locations
env_paths = [
    current_dir.parent / ".env" if current_dir.name == "notebooks" else current_dir / ".env",  # factflow/.env
    current_dir.parent.parent / ".env" if current_dir.name == "notebooks" else current_dir.parent / ".env",  # parent/.env
    Path.home() / ".env",  # home directory
]
for env_path in env_paths:
    if env_path.exists():
        load_dotenv(dotenv_path=env_path)
        print(f"Loaded .env from: {env_path}")
        break
else:
    # Try loading from current directory
    load_dotenv()

# Check for API key
if not os.getenv("GEMINI_API_KEY"):
    print("⚠️  Warning: GEMINI_API_KEY not found in environment variables.")
    print("   Please set it in your .env file or as an environment variable.")
    print("   For Kaggle notebooks, add it to Kaggle Secrets.")
else:
    print("✅ GEMINI_API_KEY found")

# Verify factflow can be imported
try:
    import factflow
    print(f"✅ factflow package found at: {factflow.__file__ if hasattr(factflow, '__file__') else 'unknown'}")
except ImportError as e:
    print(f"❌ Error importing factflow: {e}")
    print(f"Python path: {sys.path}")
    raise

# Import FactFlow components
from factflow.session.session_manager import FactFlowSessionManager
from factflow.session.memory_service import MemoryBank
from factflow.observability.logging_config import setup_logging, get_logger
from factflow.observability.tracing import TraceCollector
from factflow.observability.metrics import MetricsCollector
from factflow.tools.market_tools import get_crypto_price_data, get_stock_data, get_tradfi_context

print("✅ All imports successful!")


Current working directory: d:\Tutorials\AI Agents Intensive Course Kaghle-Google\factflow\notebooks
Added to path: d:\Tutorials\AI Agents Intensive Course Kaghle-Google
Loaded .env from: d:\Tutorials\AI Agents Intensive Course Kaghle-Google\factflow\.env
✅ GEMINI_API_KEY found
✅ factflow package found at: d:\Tutorials\AI Agents Intensive Course Kaghle-Google\factflow\__init__.py
✅ All imports successful!


## 1. Basic Agent Usage

Let's start with a simple example of using the FactFlow agent to analyze an asset.


In [10]:
# Create a session manager
session_manager = FactFlowSessionManager()

# Get a runner for a session
runner = session_manager.get_runner(session_id="demo_session")

# Run a query
print("🔍 Analyzing Ethereum...")
print("=" * 80)

query = "Assess Ethereum right now"
response = await runner.run_debug(query)

# Extract the response text
response_text = ""
if isinstance(response, list) and len(response) > 0:
    # Find the Judge Agent's output (final recommendation)
    for item in reversed(response):
        if hasattr(item, 'content') and item.content:
            if hasattr(item.content, 'parts') and item.content.parts:
                item_text = ""
                for part in item.content.parts:
                    if hasattr(part, 'text') and part.text:
                        item_text += part.text + "\n"
                
                if item_text.strip().startswith("## Sentiment-Reality Analysis"):
                    response_text = item_text
                    break
                elif not response_text and item_text.strip():
                    response_text = item_text

print(response_text)


🔍 Analyzing Ethereum...

 ### Created new session: debug_session_id

User > Assess Ethereum right now
NewsScoutAgent > Sentiment Score: -3/10

Key Themes:
*   Price decline and bearish momentum
*   Institutional investor exit
*   Key support levels being tested
*   Analyst predictions of further downside or potential reversal

Reasoning:
The latest news indicates a significant price drop for Ethereum, with multiple sources highlighting a sharp correction and bearish momentum intensifying. Institutional investors are reportedly exiting, contributing to the downward pressure and accelerating the downtrend towards lower price levels. Analysts are closely watching key support zones, with varying predictions ranging from further capitulation to potential, albeit uncertain, rebounds. Some reports mention large holders buying during the downturn, but this is contrasted with overall negative market sentiment and selling pressure. While Ethereum's underlying technology and its role in DeFi and 

JudgeAgent > ## Sentiment-Reality Analysis

**Sentiment Score:** -3/10
**Price Action:** -0.16%
**Divergence Type:** Bullish Divergence
   
**Analysis:**
The sentiment score for Ethereum is moderately negative (-3/10), driven by news of price declines, bearish momentum, and institutional investor exits. Analysts are predicting further downside, and key support levels are being tested. However, the actual price action over the last 24 hours shows a very small decline of -0.16%. This indicates that despite the negative news and sentiment, the price has remained relatively stable, or even slightly decreased, rather than experiencing a significant drop as suggested by the bearish outlook. This stability suggests that selling pressure might be weakening, or that there are underlying buyers absorbing the negative sentiment.

**Recommendation:** ACCUMULATE
   
**Reasoning:**
The combination of negative sentiment and minimal price decline suggests a potential bullish divergence. While news is 

## 2. Multiple Asset Analysis

Let's analyze multiple assets in sequence to see how the agent handles different queries.


In [11]:
# Analyze multiple assets
assets = [
    "Bitcoin",
    "Solana",
    "AAPL stock"
]

results = {}

for asset in assets:
    print(f"\n{'='*80}")
    print(f"📊 Analyzing: {asset}")
    print('='*80)
    
    query = f"Assess {asset} right now"
    response = await runner.run_debug(query)
    
    # Extract response
    response_text = ""
    if isinstance(response, list) and len(response) > 0:
        for item in reversed(response):
            if hasattr(item, 'content') and item.content:
                if hasattr(item.content, 'parts') and item.content.parts:
                    item_text = ""
                    for part in item.content.parts:
                        if hasattr(part, 'text') and part.text:
                            item_text += part.text + "\n"
                    
                    if item_text.strip().startswith("## Sentiment-Reality Analysis"):
                        response_text = item_text
                        break
                    elif not response_text and item_text.strip():
                        response_text = item_text
    
    results[asset] = response_text
    print(response_text)
    
    # Small delay to avoid rate limiting
    await asyncio.sleep(2)

print(f"\n✅ Analyzed {len(results)} assets")



📊 Analyzing: Bitcoin

 ### Continue session: debug_session_id

User > Assess Bitcoin right now
NewsScoutAgent > Sentiment Score: -6/10

Key Themes:
*   Significant price decline and bearish technicals
*   Intensified selling pressure and "extreme fear" sentiment
*   Institutional outflows from ETFs, but some accumulation at lower prices
*   Macroeconomic factors (Fed policy, liquidity) contributing to downturn

Reasoning:
Bitcoin is currently experiencing a significantly negative sentiment (-6/10). News headlines consistently report a brutal sell-off, with Bitcoin dropping to seven-month lows and major technical breakdowns. The "Crypto Fear & Greed Index" is at historic lows, indicating "extreme fear" among investors. While some institutional investors are reportedly "nibbling" at lower prices and accumulating, this is overshadowed by significant outflows from Bitcoin ETFs, particularly BlackRock's IBIT. Macroeconomic factors, such as concerns about Federal Reserve interest rate polic

JudgeAgent > ## Sentiment-Reality Analysis

**Sentiment Score:** -6/10
**Price Action:** -0.31%
**Divergence Type:** Confirmed Trend
   
**Analysis:**
The sentiment for Bitcoin is significantly negative (-6/10), characterized by reports of a brutal sell-off, bearish technicals, "extreme fear," and substantial institutional outflows from ETFs. Macroeconomic concerns are also cited as drivers of this downturn. The market analyst data shows a 24-hour price change of -0.31%. This price action aligns with the overwhelmingly negative sentiment and the bearish themes reported by the News Scout. The continued price decline, even if modest in the last 24 hours, confirms the prevailing bearish trend.

**Recommendation:** SELL
   
**Reasoning:**
The sentiment and price action are in alignment, indicating a confirmed bearish trend. The significant negative sentiment, coupled with ongoing outflows and bearish technicals, suggests that further downside is probable in the short term. Therefore, a sel

JudgeAgent > ## Sentiment-Reality Analysis

**Sentiment Score:** -7/10
**Price Action:** -0.46%
**Divergence Type:** Confirmed Trend
   
**Analysis:**
The sentiment for Solana is severely negative at -7/10. Key themes include a significant price drop below support levels, high selling volume, bearish technical indicators, and potential concerns about network congestion and transaction fees. The sentiment also highlights Solana's underperformance relative to the broader market. The Market Analyst Agent reports a 24-hour price change of -0.46% for Solana. This price action aligns closely with the strong bearish sentiment and news themes. The loss of support and high selling volume, reflected in the negative price change, confirms the prevailing negative trend.

**Recommendation:** SELL
   
**Reasoning:**
There is a clear alignment between the very negative sentiment (-7/10) and the actual price action (-0.46% decline), indicating a confirmed bearish trend. The combination of technical br

JudgeAgent > ## Sentiment-Reality Analysis

**Sentiment Score:** -4/10
**Price Action:** -0.88%
**Divergence Type:** Confirmed Trend
   
**Analysis:**
The sentiment for AAPL stock is moderately negative (-4/10). This sentiment is driven by significant headwinds, including ongoing antitrust investigations in the US, regulatory scrutiny in Europe over App Store policies, and concerns about slowing iPhone demand, particularly in China. While there are some minor positive analyst movements, they are overshadowed by these substantial negative factors. The market data shows a 24-hour price change of -0.88%. This price action aligns with the negative sentiment, indicating that investors are reacting to the news and regulatory concerns by selling the stock.

**Recommendation:** SELL
   
**Reasoning:**
The sentiment and price action for AAPL are aligned, indicating a confirmed negative trend. The moderately negative sentiment (-4/10) is supported by a -0.88% decline in the stock price over the 

## 3. Memory Bank Usage

The Memory Bank stores historical analyses and allows the agent to learn from past patterns.


In [12]:
# Initialize Memory Bank
memory = MemoryBank(storage_path="demo_memory.json")

# Store some sample analyses
print("💾 Storing analyses in Memory Bank...")

memory.store_analysis(
    asset="ethereum",
    sentiment_score=-8.0,
    price_change=-0.5,
    recommendation="HOLD",
    divergence_type="bullish",
)

memory.store_analysis(
    asset="bitcoin",
    sentiment_score=7.5,
    price_change=2.3,
    recommendation="BUY",
    divergence_type=None,
)

memory.store_analysis(
    asset="solana",
    sentiment_score=9.0,
    price_change=-1.2,
    recommendation="SELL",
    divergence_type="bearish",
)

print("✅ Analyses stored")


💾 Storing analyses in Memory Bank...
✅ Analyses stored


In [13]:
# Retrieve asset history
print("\n📜 Asset History:")
print("=" * 80)

ethereum_history = memory.get_asset_history("ethereum", limit=5)
print(f"\nEthereum History ({len(ethereum_history)} analyses):")
for analysis in ethereum_history:
    print(f"  - {analysis['timestamp']}: Sentiment {analysis['sentiment_score']}/10, "
          f"Price {analysis['price_change']:.2f}%, "
          f"Recommendation: {analysis['recommendation']}")

bitcoin_history = memory.get_asset_history("bitcoin", limit=5)
print(f"\nBitcoin History ({len(bitcoin_history)} analyses):")
for analysis in bitcoin_history:
    print(f"  - {analysis['timestamp']}: Sentiment {analysis['sentiment_score']}/10, "
          f"Price {analysis['price_change']:.2f}%, "
          f"Recommendation: {analysis['recommendation']}")



📜 Asset History:

Ethereum History (1 analyses):
  - 2025-11-23T00:46:00.426259: Sentiment -8.0/10, Price -0.50%, Recommendation: HOLD

Bitcoin History (1 analyses):
  - 2025-11-23T00:46:00.427255: Sentiment 7.5/10, Price 2.30%, Recommendation: BUY


In [14]:
# Get divergence patterns
print("\n🔍 Divergence Patterns:")
print("=" * 80)

patterns = memory.get_divergence_patterns(limit=10)
print(f"\nFound {len(patterns)} divergence patterns:")

for pattern in patterns:
    print(f"  - {pattern['asset']}: {pattern['divergence_type']} divergence "
          f"(Sentiment: {pattern['sentiment_score']}, Price: {pattern['price_change']:.2f}%)")



🔍 Divergence Patterns:

Found 2 divergence patterns:
  - solana: bearish divergence (Sentiment: 9.0, Price: -1.20%)
  - ethereum: bullish divergence (Sentiment: -8.0, Price: -0.50%)


In [15]:
# Get context summary
print("\n📋 Context Summary:")
print("=" * 80)

summary = memory.get_context_summary()
print(summary)

ethereum_summary = memory.get_context_summary(asset="ethereum")
print(f"\nEthereum-specific summary:\n{ethereum_summary}")



📋 Context Summary:
Memory Bank Summary: 3 total analyses, 2 divergence patterns detected.

Ethereum-specific summary:
Historical analysis for ethereum:
- 2025-11-23T00:46:00.426259: Sentiment -8.0/10, Price -0.50%, Recommendation: HOLD



## 4. Observability Features

FactFlow includes comprehensive observability features: logging, tracing, and metrics collection.


In [16]:
# Setup logging
logger = setup_logging(log_level="INFO", log_file="demo_factflow.log")
logger.info("Starting FactFlow demo with observability")

# Initialize observability components
trace_collector = TraceCollector(trace_file="demo_traces.json")
metrics_collector = MetricsCollector(metrics_file="demo_metrics.json")

print("✅ Observability components initialized")


2025-11-23 00:46:47 - factflow - INFO - Logging configured successfully
2025-11-23 00:46:47 - factflow - INFO - Starting FactFlow demo with observability
✅ Observability components initialized


In [17]:
# Run a query with full observability
query = "Assess Bitcoin right now"
session_id = "observability_demo"

# Start trace
trace_id = trace_collector.start_trace(session_id, query)
logger.info(f"Started trace: {trace_id}")

print(f"🔍 Running query with trace ID: {trace_id}")
print("=" * 80)

# Get runner
runner = session_manager.get_runner(session_id=session_id)

# Run query
response = await runner.run_debug(query)

# Extract response
response_text = ""
if isinstance(response, list) and len(response) > 0:
    for item in reversed(response):
        if hasattr(item, 'content') and item.content:
            if hasattr(item.content, 'parts') and item.content.parts:
                item_text = ""
                for part in item.content.parts:
                    if hasattr(part, 'text') and part.text:
                        item_text += part.text + "\n"
                
                if item_text.strip().startswith("## Sentiment-Reality Analysis"):
                    response_text = item_text
                    break
                elif not response_text and item_text.strip():
                    response_text = item_text

print(response_text)

# End trace
trace = trace_collector.end_trace(final_output=response_text)
print(f"\n✅ Trace completed: {trace['trace_id']}")
print(f"   Duration: {trace.get('total_duration_ms', 0):.2f} ms")

# Record metrics
metrics_collector.record_query(success=True)
print("✅ Metrics recorded")


2025-11-23 00:47:54 - factflow - INFO - Started trace: trace_20251123_004754_027965
🔍 Running query with trace ID: trace_20251123_004754_027965

 ### Created new session: debug_session_id

User > Assess Bitcoin right now
NewsScoutAgent > Sentiment Score: -6/10
Key Themes:
*   Recent price decline and bearish sentiment.
*   Impact of stock market volatility and macroeconomic factors (interest rates).
*   Investor fear and capitulation.
*   Outflows from Bitcoin ETFs.

Reasoning:
The recent news indicates a significant downturn for Bitcoin, with prices dropping to seven-month lows and the "fear and greed" index reaching historic lows. This is attributed to a combination of factors, including a broader market sell-off influenced by volatility in tech stocks and concerns about potential interest rate hikes by the Federal Reserve, which reduces market liquidity. Furthermore, outflows from Bitcoin ETFs and reports of panic selling by investors highlight a prevailing negative sentiment. While

JudgeAgent > ## Sentiment-Reality Analysis

**Sentiment Score:** -6/10
**Price Action:** -0.27%

**Divergence Type:** None

**Analysis:**
The sentiment score of -6/10 indicates a bearish outlook, driven by recent price declines, investor fear, and macroeconomic concerns such as potential interest rate hikes. This aligns with the slight negative price action observed in Bitcoin over the last 24 hours (-0.27%). The market appears to be reacting as expected to the prevailing negative sentiment and news themes. There is no significant divergence between the sentiment and the price movement, suggesting that the current price action is a reflection of the reported sentiment.

**Recommendation:** HOLD

**Reasoning:**
While the sentiment is negative and the price has seen a minor decline, it's not a situation that warrants a strong buy or sell signal on its own. The -6/10 sentiment score, coupled with a small negative price change, suggests a market that is experiencing downward pressure but h

In [18]:
# View trace details
print("\n📊 Trace Details:")
print("=" * 80)

if trace:
    print(f"Trace ID: {trace['trace_id']}")
    print(f"Session ID: {trace['session_id']}")
    print(f"Query: {trace['user_query']}")
    print(f"Start Time: {trace['start_time']}")
    print(f"End Time: {trace['end_time']}")
    print(f"Duration: {trace.get('total_duration_ms', 0):.2f} ms")
    print(f"\nAgents Executed: {len(trace.get('agents', []))}")
    for agent in trace.get('agents', []):
        print(f"  - {agent['agent_name']}")
        if 'tool_calls' in agent and agent['tool_calls']:
            print(f"    Tools used: {[tc['tool_name'] for tc in agent['tool_calls']]}")
    
    print(f"\nTotal Tool Calls: {len(trace.get('tool_calls', []))}")
    if trace.get('errors'):
        print(f"Errors: {len(trace['errors'])}")
    else:
        print("Errors: None ✅")



📊 Trace Details:
Trace ID: trace_20251123_004754_027965
Session ID: observability_demo
Query: Assess Bitcoin right now
Start Time: 2025-11-23T00:47:54.027965
End Time: 2025-11-23T00:48:04.952625
Duration: 10924.66 ms

Agents Executed: 0

Total Tool Calls: 0
Errors: None ✅


In [19]:
# Record some sample metrics
metrics_collector.record_sentiment_score(-8.0, asset="bitcoin")
metrics_collector.record_sentiment_score(7.5, asset="ethereum")
metrics_collector.record_divergence("bullish", -8.0, -0.5, "bitcoin")
metrics_collector.record_tool_call("get_crypto_price_data", 250.5)
metrics_collector.record_tool_call("get_stock_data", 180.2)
metrics_collector.record_agent_execution("NewsScoutAgent", 1200.0)
metrics_collector.record_agent_execution("MarketAnalystAgent", 800.0)
metrics_collector.record_agent_execution("JudgeAgent", 600.0)

# Get metrics summary
print("\n📈 Metrics Summary:")
print("=" * 80)

summary = metrics_collector.get_summary()
print(f"Total Queries: {summary['total_queries']}")
print(f"Successful Queries: {summary['successful_queries']}")
print(f"Success Rate: {summary['success_rate']:.2%}")

if 'sentiment_stats' in summary:
    print(f"\nSentiment Statistics:")
    print(f"  Count: {summary['sentiment_stats']['count']}")
    print(f"  Mean: {summary['sentiment_stats']['mean']:.2f}")
    print(f"  Min: {summary['sentiment_stats']['min']:.2f}")
    print(f"  Max: {summary['sentiment_stats']['max']:.2f}")

if 'divergence_detection_rate' in summary:
    print(f"\nDivergence Detection Rate: {summary['divergence_detection_rate']:.2%}")

if summary.get('avg_tool_latencies'):
    print(f"\nAverage Tool Latencies:")
    for tool, latency in summary['avg_tool_latencies'].items():
        print(f"  {tool}: {latency:.2f} ms")

if summary.get('avg_agent_times'):
    print(f"\nAverage Agent Execution Times:")
    for agent, time in summary['avg_agent_times'].items():
        print(f"  {agent}: {time:.2f} ms")



📈 Metrics Summary:
Total Queries: 1
Successful Queries: 1
Success Rate: 100.00%

Sentiment Statistics:
  Count: 2
  Mean: -0.25
  Min: -8.00
  Max: 7.50

Divergence Detection Rate: 100.00%

Average Tool Latencies:
  get_crypto_price_data: 250.50 ms
  get_stock_data: 180.20 ms

Average Agent Execution Times:
  NewsScoutAgent: 1200.00 ms
  MarketAnalystAgent: 800.00 ms
  JudgeAgent: 600.00 ms


## 5. Tool Testing

Let's test the individual tools that the agent uses to fetch market data.


In [20]:
# Test crypto price data tool
print("🪙 Testing Crypto Price Data Tool:")
print("=" * 80)

crypto_result = get_crypto_price_data("ethereum")
print(f"\nAsset: {crypto_result.get('asset', 'N/A')}")
if crypto_result.get('status') == 'success':
    print(f"Current Price: ${crypto_result.get('current_price', 0):,.2f}")
    print(f"24h Change: {crypto_result.get('price_change_24h', 0):.2f}%")
    print(f"24h Volume: ${crypto_result.get('volume_24h', 0):,.2f}")
    print(f"Market Cap: ${crypto_result.get('market_cap', 0):,.2f}")
else:
    print(f"Error: {crypto_result.get('error', 'Unknown error')}")


🪙 Testing Crypto Price Data Tool:

Asset: ethereum
Current Price: $2,742.36
24h Change: 0.05%
24h Volume: $15,493,836,209.44
Market Cap: $331,050,942,002.93


In [21]:
# Test stock data tool
print("\n📈 Testing Stock Data Tool:")
print("=" * 80)

stock_result = get_stock_data("AAPL")
print(f"\nSymbol: {stock_result.get('symbol', 'N/A')}")
if stock_result.get('status') == 'success':
    print(f"Current Price: ${stock_result.get('current_price', 0):,.2f}")
    print(f"24h Change: {stock_result.get('price_change_24h', 0):.2f}%")
    print(f"Volume: {stock_result.get('volume', 0):,.0f}")
    print(f"Market Cap: ${stock_result.get('market_cap', 0):,.2f}")
else:
    print(f"Error: {stock_result.get('error', 'Unknown error')}")



📈 Testing Stock Data Tool:

Symbol: AAPL
Current Price: $271.50
24h Change: -0.88%
Volume: 4,300,075
Market Cap: $4,029,017,227,264.00


In [22]:
# Test TradFi context tool
print("\n🌍 Testing TradFi Context Tool:")
print("=" * 80)

tradfi_result = get_tradfi_context()
if tradfi_result.get('status') == 'success':
    print(f"S&P 500 24h Change: {tradfi_result.get('sp500_change', 0):.2f}%")
    print(f"NASDAQ 24h Change: {tradfi_result.get('nasdaq_change', 0):.2f}%")
    print(f"Correlation Indicator: {tradfi_result.get('correlation_indicator', 'N/A')}")
else:
    print(f"Error: {tradfi_result.get('error', 'Unknown error')}")



🌍 Testing TradFi Context Tool:
S&P 500 24h Change: -2.35%
NASDAQ 24h Change: -3.58%
Correlation Indicator: positive


## 6. Session Management

FactFlow supports multi-turn conversations through session management. Let's see how the agent maintains context across multiple queries.


In [23]:
# Create a new session for multi-turn conversation
session_id = "multi_turn_demo"
runner = session_manager.get_runner(session_id=session_id)

print("💬 Multi-Turn Conversation Demo:")
print("=" * 80)

# First query
print("\n1️⃣ First Query:")
query1 = "Assess Ethereum right now"
print(f"Query: {query1}")

response1 = await runner.run_debug(query1)
response_text1 = ""
if isinstance(response1, list) and len(response1) > 0:
    for item in reversed(response1):
        if hasattr(item, 'content') and item.content:
            if hasattr(item.content, 'parts') and item.content.parts:
                item_text = ""
                for part in item.content.parts:
                    if hasattr(part, 'text') and part.text:
                        item_text += part.text + "\n"
                
                if item_text.strip().startswith("## Sentiment-Reality Analysis"):
                    response_text1 = item_text
                    break
                elif not response_text1 and item_text.strip():
                    response_text1 = item_text

print(response_text1[:500] + "..." if len(response_text1) > 500 else response_text1)


💬 Multi-Turn Conversation Demo:

1️⃣ First Query:
Query: Assess Ethereum right now

 ### Created new session: debug_session_id

User > Assess Ethereum right now
NewsScoutAgent > - Sentiment Score: -3/10
- Key Themes: Market downturn, institutional investor exit, selling pressure, liquidations, price prediction for further losses.
- Reasoning: The majority of headlines indicate a negative sentiment for Ethereum, with reports of a sell-off, intensifies liquidations, and institutional investors exiting the market. Price predictions suggest further losses, with ETH falling below key support levels. While there's mention of a whale aggressively buying Ethereum, this is overshadowed by the broader negative market sentiment and institutional pullback.


MarketAnalystAgent > Ethereum is currently trading at $2742.40, with a 24-hour price change of -0.73%. The trading volume is $15,468,718,450.39. The market capitalization is $331,050,942,002.93.

The traditional markets (S&P 500 and NASDAQ) experienced a negative change of -2.35% and -3.58% respectively. The correlation indicator suggests a positive correlation between Ethereum and the traditional markets.

**Market Analysis:**

*   **Price Action:** Ethereum has shown a moderate downward movement in the last 24 hours.
*   **Volume:** The trading volume appears to be within an average range, not indicating significant buying or selling pressure.
*   **Market Correlation:** Ethereum's movement is currently aligned with the broader traditional financial markets, which are also experiencing a downturn. This suggests that the current price action might be influenced by macroeconomic factors rather than purely crypto-specific events.

The provided news sentiment aligns with this analysis, i

In [24]:
# Follow-up query (agent should remember context)
print("\n2️⃣ Follow-up Query:")
query2 = "What about Bitcoin?"
print(f"Query: {query2}")
print("(The agent should maintain context from the previous query)")

response2 = await runner.run_debug(query2)
response_text2 = ""
if isinstance(response2, list) and len(response2) > 0:
    for item in reversed(response2):
        if hasattr(item, 'content') and item.content:
            if hasattr(item.content, 'parts') and item.content.parts:
                item_text = ""
                for part in item.content.parts:
                    if hasattr(part, 'text') and part.text:
                        item_text += part.text + "\n"
                
                if item_text.strip().startswith("## Sentiment-Reality Analysis"):
                    response_text2 = item_text
                    break
                elif not response_text2 and item_text.strip():
                    response_text2 = item_text

print(response_text2[:500] + "..." if len(response_text2) > 500 else response_text2)



2️⃣ Follow-up Query:
Query: What about Bitcoin?
(The agent should maintain context from the previous query)

 ### Continue session: debug_session_id

User > What about Bitcoin?
NewsScoutAgent > - Sentiment Score: -5/10
- Key Themes: Market downturn, selling pressure, liquidations, correlation with tech stocks, technical resistance, institutional outflows from ETFs.
- Reasoning: The recent news indicates a negative sentiment for Bitcoin, driven by a significant market downturn, intensifying selling pressure, and liquidations reaching billions. Bitcoin is experiencing a correlation with the tech sector, which is also facing weakness, and is trading at seven-month lows. Technical indicators suggest potential further downside, with a bearish SuperTrend signal and a macro Head & Shoulders pattern observed. While there are mentions of long-term holders buying at lower prices and some ETF inflows, these are currently outweighed by the overwhelming negative news regarding price declines, liqu

MarketAnalystAgent > Bitcoin is currently priced at $84,345, showing a 24-hour price change of -0.51%. The trading volume is substantial at $45,731,177,417.19, and its market capitalization stands at $1,684,683,827,569.03.

In the broader market context, both the S&P 500 and NASDAQ have seen significant downturns, with changes of -2.35% and -3.58% respectively. The `tradfi_context` indicates a positive correlation between Bitcoin and these traditional markets.

**Market Analysis:**

*   **Price Action:** Bitcoin has experienced a moderate price decrease in the last 24 hours.
*   **Volume:** The trading volume is significantly above average, suggesting heightened activity and potentially increased investor interest despite the price dip.
*   **Market Correlation:** Bitcoin's movement is currently aligned with the traditional financial markets, which are also experiencing a downturn. This indicates that broader economic factors may be influencing Bitcoin's price.

The sentiment analysis 

## Summary

This notebook demonstrated:

✅ **Basic Agent Usage** - Running queries with the FactFlow agent  
✅ **Multiple Asset Analysis** - Analyzing different assets in sequence  
✅ **Memory Bank Usage** - Storing and retrieving historical analyses  
✅ **Observability Features** - Logging, tracing, and metrics collection  
✅ **Tool Testing** - Testing individual market data tools  
✅ **Session Management** - Multi-turn conversations with context  

### Next Steps

- Try analyzing your own assets
- Experiment with different query formats
- Explore the Memory Bank patterns
- Review the trace and metrics files
- Check out the full documentation in `USAGE.md`

### Notes

- Make sure you have set your `GEMINI_API_KEY` before running
- Some cells may take 30-60 seconds to complete (API calls)
- The notebook uses async/await, so run cells in order
- For Kaggle notebooks, add your API key to Kaggle Secrets
